# SupplyForge AI 2.0: KAN-27 - Daily Calendar-SKU Grid Generation

Tài liệu này thực hiện tạo ma trận lưới Daily × SKU hoàn chỉnh từ dữ liệu Silver Layer () nhằm biến đổi chuỗi giao dịch ngắt quãng thành chuỗi thời gian liên tục (continuous time-series).


### Quy tắc nghiệp vụ và Tiêu chuẩn Nghiệm thu (Acceptance Criteria):
1. **date_range liên tục**: Xây dựng dải ngày liên tục từ ngày giao dịch nhỏ nhất đến lớn nhất.
2. **Active SKUs**: Xác định toàn bộ SKU xuất hiện trong dữ liệu sạch Silver Layer.
3. **Cartesian Product**: Dùng  tạo đầy đủ tổ hợp (date, itemId).
4. **Left Join & Imputation**: Điền các ngày không phát sinh nhu cầu bằng  (Zero sales).
5. **Gold Layer Output**: Lưu kết quả tại .


In [1]:
import pandas as pd
import numpy as np
import os

PROJECT_ROOT = "/home/nguyen-dai-tung/supplyforge-ai"
silver_items_path = os.path.join(PROJECT_ROOT, "data", "silver", "order_items_clean.parquet")

print("Đang tải dữ liệu order_items sạch từ Silver Layer...")
df_order_items_raw = pd.read_parquet(silver_items_path)
print(f"Tổng số dòng order_items sạch: {len(df_order_items_raw):,}")


Đang tải dữ liệu order_items sạch từ Silver Layer...
Tổng số dòng order_items sạch: 559,254


In [2]:
print("Đang chuẩn bị dữ liệu bán hàng hàng ngày...")
df_order_items = df_order_items_raw[["time", "itemId", "quantity"]].copy()
df_order_items["date"] = pd.to_datetime(df_order_items["time"]).dt.date
print(f"Tổng số giao dịch chi tiết: {len(df_order_items):,}")
print(f"Tổng lượng bán hàng trước khi gộp nhóm: {df_order_items['quantity'].sum():,}")


Đang chuẩn bị dữ liệu bán hàng hàng ngày...


Tổng số giao dịch chi tiết: 559,254
Tổng lượng bán hàng trước khi gộp nhóm: 947,088


In [3]:
print("Đang gộp lượng bán hàng theo Ngày và SKU...")
df_daily_agg = df_order_items.groupby(["date", "itemId"])["quantity"].sum().reset_index()
print(f"Tổng số dòng sau khi gộp nhóm: {len(df_daily_agg):,}")
print(f"Tổng lượng bán hàng sau khi gộp nhóm: {df_daily_agg['quantity'].sum():,}")


Đang gộp lượng bán hàng theo Ngày và SKU...
Tổng số dòng sau khi gộp nhóm: 89,049
Tổng lượng bán hàng sau khi gộp nhóm: 947,088


In [4]:
min_date = df_daily_agg["date"].min()
max_date = df_daily_agg["date"].max()
date_range = pd.date_range(start=min_date, end=max_date, freq="D").date
active_skus = df_daily_agg["itemId"].unique()

print(f"Phạm vi ngày liên tục: Từ {min_date} đến {max_date} ({len(date_range)} ngày)")
print(f"Số lượng Active SKUs: {len(active_skus)} SKU")
print(f"Số dòng dự kiến trong ma trận đầy đủ (Days * Active SKUs): {len(date_range) * len(active_skus):,}")


Phạm vi ngày liên tục: Từ 2021-04-22 đến 2025-12-15 (1699 ngày)
Số lượng Active SKUs: 940 SKU
Số dòng dự kiến trong ma trận đầy đủ (Days * Active SKUs): 1,597,060


In [5]:
print("Đang tạo lưới rỗng bằng pd.MultiIndex.from_product...")
grid_index = pd.MultiIndex.from_product([date_range, active_skus], names=["date", "itemId"])
df_grid = pd.DataFrame(index=grid_index).reset_index()

print("Đang ghép nối doanh số thực tế và điền khuyết lượng bán bằng 0...")
df_calendar = pd.merge(df_grid, df_daily_agg, on=["date", "itemId"], how="left")
df_calendar["quantity"] = df_calendar["quantity"].fillna(0).astype("int32")
print(f"Hoàn thành ghép lưới. Tổng số dòng thực tế: {len(df_calendar):,}")


Đang tạo lưới rỗng bằng pd.MultiIndex.from_product...
Đang ghép nối doanh số thực tế và điền khuyết lượng bán bằng 0...


Hoàn thành ghép lưới. Tổng số dòng thực tế: 1,597,060


In [6]:
print("=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===")
expected_rows = len(date_range) * len(active_skus)
actual_rows = len(df_calendar)
assert actual_rows == expected_rows, f"Lỗi: Số dòng thực tế ({actual_rows}) khác lý thuyết ({expected_rows})"
print(f"1. Tổng số dòng khớp lý thuyết: Đạt ({actual_rows:,} dòng)")

duplicates_cnt = df_calendar.duplicated(subset=["date", "itemId"]).sum()
assert duplicates_cnt == 0, f"Lỗi: Phát hiện {duplicates_cnt} bản ghi trùng lặp"
print("2. Không có trùng lặp (date, itemId) trong lưới: Đạt")

null_qty_cnt = df_calendar["quantity"].isnull().sum()
assert null_qty_cnt == 0, "Lỗi: Phát hiện giá trị Null trong cột quantity"
print("3. Không có giá trị null trong cột quantity: Đạt")

sum_before = df_daily_agg["quantity"].sum()
sum_after = df_calendar["quantity"].sum()
assert sum_before == sum_after, f"Lỗi: Lượng bán thay đổi trước ({sum_before}) và sau ({sum_after})"
print(f"4. Tổng lượng bán hàng được bảo toàn: Đạt ({sum_after:,} sản phẩm)")

uniq_dates = df_calendar["date"].nunique()
assert uniq_dates == len(date_range), "Lỗi: Số lượng ngày không khớp"
print(f"5. Lưới chứa đầy đủ ngày, không bị khuyết: Đạt ({uniq_dates} ngày)")

zero_sales_ratio = (df_calendar["quantity"] == 0).mean() * 100
print(f"6. Tỷ lệ Zero-Sales trong lưới: {zero_sales_ratio:.4f}%")


=== THỰC THI KIỂM CHỨNG & XÁC THỰC (VERIFICATION) ===
1. Tổng số dòng khớp lý thuyết: Đạt (1,597,060 dòng)
2. Không có trùng lặp (date, itemId) trong lưới: Đạt
3. Không có giá trị null trong cột quantity: Đạt
4. Tổng lượng bán hàng được bảo toàn: Đạt (947,088 sản phẩm)
5. Lưới chứa đầy đủ ngày, không bị khuyết: Đạt (1699 ngày)
6. Tỷ lệ Zero-Sales trong lưới: 94.4242%


In [7]:
print("=== LƯU TRỮ DỮ LIỆU ĐÃ GHÉP LƯỚI VÀO GOLD LAYER ===")
gold_dir = os.path.join(PROJECT_ROOT, "data", "gold")
os.makedirs(gold_dir, exist_ok=True)
calendar_output_parquet = os.path.join(gold_dir, "calendar_daily_sales.parquet")
calendar_output_csv = os.path.join(gold_dir, "calendar_daily_sales.csv")

df_calendar.to_parquet(calendar_output_parquet, index=False)
df_calendar.to_csv(calendar_output_csv, index=False)
print(f"Đã lưu lưới Daily * SKU thành công tại Gold Layer: {calendar_output_parquet}")


=== LƯU TRỮ DỮ LIỆU ĐÃ GHÉP LƯỚI VÀO GOLD LAYER ===


Đã lưu lưới Daily * SKU thành công tại Gold Layer: /home/nguyen-dai-tung/supplyforge-ai/data/gold/calendar_daily_sales.parquet
